In [1]:
# %pip install spacy

In [3]:
import re
import os
from typing import List, Dict, Tuple, Optional
import spacy
from spacy.tokens import Doc, Span
import pandas as pd
from PyPDF2 import PdfReader

In [4]:
zeroshot_path = "reports/Report Zero Shot.txt"
with open(zeroshot_path, "r", encoding="utf-8", errors="ignore") as f:
    zeroshot_report_text = f.read()

cot_path = "reports/Report COT.txt"
with open(cot_path, "r", encoding="utf-8", errors="ignore") as f:
    cot_report_text = f.read()

fewshot_path = "reports/Report Few Shot.txt"
with open(fewshot_path, "r", encoding="utf-8", errors="ignore") as f:
    fewshot_report_text = f.read()



article1 = "publication articles/Motility Software Solutions.pdf"
article2 = "publication articles/Hyundai.pdf"
article3 = "publication articles/Electronics manufacturer.txt"
article4 = "publication articles/Nidec.pdf"
article5 = "publication articles/Jaguar.pdf"

article_paths = [article1, article2, article3, article4, article5]

In [5]:
# loading spaCy;
def load_spacy_model():
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("⚠️ spaCy model not found. Falling back to blank English.")
        nlp = spacy.blank("en")
        nlp.add_pipe("sentencizer")
    return nlp

nlp = load_spacy_model()

In [6]:
def span_from_token_subtree(doc: Doc, token) -> Span:
    idxs = [t.i for t in token.subtree]
    return doc[min(idxs): max(idxs) + 1]

def split_compound(sentence: Span) -> List[str]:
    claims = set()
    text = sentence.text.strip()

    if not text:
        return []

    claims.add(text)

    # Semicolon splits
    for part in re.split(r";+", text):
        if len(part.split()) >= 3:
            claims.add(part.strip())

    if "parser" in nlp.pipe_names:
        for token in sentence:
            if token.dep_ in {"conj", "advcl", "ccomp", "xcomp", "acl"}:
                span = span_from_token_subtree(sentence.doc, token)
                claims.add(span.text.strip(" ,"))

        for conj in (" and ", " or "):
            parts = [p.strip() for p in text.split(conj)]
            if len(parts) > 1:
                claims.update(p for p in parts if len(p.split()) >= 3)
    else:
        for sep in (" and ", " or ", ","):
            claims.update(
                p.strip() for p in text.split(sep) if len(p.split()) >= 3
            )

    def clean(s: str) -> str:
        return re.sub(r"\s+", " ", s).strip(" .,:;")

    seen = set()
    output = []
    for c in map(clean, claims):
        key = c.lower()
        if key not in seen:
            seen.add(key)
            output.append(c)

    return output

def extract_claims(doc: Doc) -> List[Dict]:
    rows = []
    for i, sent in enumerate(doc.sents):
        for claim in split_compound(sent):
            rows.append({"claim": claim, "sent_index": i})
    return rows

In [7]:
# article loading (.txt & .pdf)

def read_text_file(path: str) -> str:
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

def read_pdf_file(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def load_article_text(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".txt":
        return read_text_file(path)
    if ext == ".pdf":
        return read_pdf_file(path)
    raise ValueError(f"Unsupported file type: {path}")

In [8]:
# Corpus building

CONTENT_POS = {"NOUN", "VERB", "ADJ", "PROPN"}

def content_lemmas(doc: Doc) -> List[str]:
    return [
        t.lemma_.lower()
        for t in doc
        if not t.is_stop and not t.is_punct and (t.pos_ in CONTENT_POS or t.is_alpha)
    ]

def bigrams(xs: List[str]) -> set:
    return set(zip(xs, xs[1:]))

def build_corpus(article_paths: List[str]) -> List[Dict]:
    corpus = []

    for path in article_paths:
        if not os.path.exists(path):
            print(f"⚠️ Missing file: {path}")
            continue

        text = load_article_text(path)
        doc = nlp(text)
        sents = list(doc.sents)

        sent_lemmas = []
        sent_bigrams = []

        for s in sents:
            lemmas = content_lemmas(s)
            sent_lemmas.append(set(lemmas))
            sent_bigrams.append(bigrams(lemmas))

        corpus.append({
            "article_path": path,
            "doc": doc,
            "sents": sents,
            "lemmas": sent_lemmas,
            "bigrams": sent_bigrams
        })

    print(f"Loaded {len(corpus)} articles.")
    return corpus

In [9]:
# Similarity scoring

def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def score_similarity(
    claim_lemmas: set,
    claim_bigrams: set,
    sent_lemmas: set,
    sent_bigrams: set
) -> float:
    base = jaccard(claim_lemmas, sent_lemmas)
    bonus = jaccard(claim_bigrams, sent_bigrams) * 0.5
    return base + bonus

def best_support_span(
    claim_lemmas: set,
    claim_bigrams: set,
    article: Dict,
    context_window: int = 1
) -> Dict:
    best_score = 0.0
    best_i = None

    for i, s_lemmas in enumerate(article["lemmas"]):
        sc = score_similarity(
            claim_lemmas,
            claim_bigrams,
            s_lemmas,
            article["bigrams"][i]
        )
        if sc > best_score:
            best_score = sc
            best_i = i

    if best_i is None:
        return {
            #"article_path": article["article_path"],
            "span_text": "",
            "score": 0.0,
            "sent_index": None
        }

    start = max(0, best_i - context_window)
    end = min(len(article["sents"]) - 1, best_i + context_window)

    span_text = " ".join(article["sents"][j].text for j in range(start, end + 1))

    return {
        #"article_path": article["article_path"],
        "span_text": span_text,
        "score": round(best_score, 4),
        "sent_index": best_i
    }


# Factual claim detection
NON_FACTUAL_MARKERS = {
    "should", "could", "may", "might", "recommend",
    "suggest", "believe", "opinion", "likely"
}

def is_factual_claim(claim: str) -> bool:
    doc = nlp(claim)

    if not any(t.pos_ == "VERB" for t in doc):
        return False

    if any(t.lemma_.lower() in NON_FACTUAL_MARKERS for t in doc):
        return False

    content = [
        t for t in doc
        if t.pos_ in CONTENT_POS and not t.is_stop
    ]

    return len(content) >= 3

In [18]:
# Faithfulness computation
SUPPORTED_THRESHOLD = 0.06 #0.25

def is_supported(claim: str, corpus: List[Dict]) -> bool:
    doc = nlp(claim)
    lemmas = content_lemmas(doc)
    claim_lemmas = set(lemmas)
    claim_bigrams = bigrams(lemmas)

    for article in corpus:
        best = best_support_span(
            claim_lemmas,
            claim_bigrams,
            article
        )
        if best["score"] >= SUPPORTED_THRESHOLD:
            return True

    return False

def faithfulness_score(
    report_text: str,
    article_paths: List[str]
) -> Dict:

    if os.path.exists(report_text):
        raise ValueError(
            "report_text is a file path. Read the file contents first."
        )

    report_doc = nlp(report_text)
    claims = extract_claims(report_doc)
    corpus = build_corpus(article_paths)

    factual_claims = []
    supported_claims = []

    for c in claims:
        claim_text = c["claim"]

        if not is_factual_claim(claim_text):
            continue

        factual_claims.append(claim_text)

        if is_supported(claim_text, corpus):
            supported_claims.append(claim_text)

    total = len(factual_claims)
    supported = len(supported_claims)

    return {
        "faithfulness_score": round(supported / total, 3) if total else 0.0,
        "total_factual_claims": total,
        "supported_claims": supported,
        "unsupported_claims": list(set(factual_claims) - set(supported_claims))
    }

In [19]:
ZSresult = faithfulness_score(zeroshot_report_text, article_paths)
ZSresult

Loaded 5 articles.


{'faithfulness_score': 0.61,
 'total_factual_claims': 77,
 'supported_claims': 47,
 'unsupported_claims': ['targeting the airline sector',
  'disrupting retail and production activities',
  '- Provide cybersecurity training to employees. - Ensure sensitive data is encrypted both at rest and in transit. - Develop and regularly update an incident response plan',
  '- Develop and regularly update an incident response plan',
  'Jaguar Land Rover Ransomware Cyberattack - Threat: Ransomware cyberattack disrupting retail',
  'Nidec Corporation Cyberattack - Threat: Cyberattack involving the theft of sensitive data. - Affected Data: Internal documents, business partner letters, procurement documents, labor safety policies, contracts. - Timeline: Early June 2024. - Threat Actors: 8BASE',
  'Nidec Corporation Cyberattack - Mitigations: Enhance VPN security, regular security audits, employee training, data encryption, incident response plan, monitor dark web. - Recommendations: - Implement multi-

In [20]:
FSresult = faithfulness_score(fewshot_report_text, article_paths)
FSresult

Loaded 5 articles.


{'faithfulness_score': 0.639,
 'total_factual_claims': 83,
 'supported_claims': 53,
 'unsupported_claims': ['to restrict unauthorized access to their credit files',
  '- Work at pace to restart global applications in a controlled manner and monitor for any signs of data theft',
  'Motility Software Solutions Data Breach - Threat Description: Motility Software Solutions, a subsidiary of Reynolds and Reynolds, experienced a data breach that exposed sensitive information of 760,000 customers',
  'causing operational outages',
  'Conclusion The recent cybersecurity incidents highlight the critical need for organizations to enhance their cybersecurity measures, implement robust backup and recovery solutions, and stay vigilant against evolving threats',
  'relevant authorities. - Affected individuals can place fraud alerts on their credit files or establish security freezes to restrict unauthorized access to their credit files',
  "Compromised data includes names, dates of birth, driver's li

In [21]:
COTresult = faithfulness_score(cot_report_text, article_paths)
COTresult

Loaded 5 articles.


{'faithfulness_score': 0.58,
 'total_factual_claims': 69,
 'supported_claims': 40,
 'unsupported_claims': ['working to restart them in a controlled manner, with no evidence of customer data theft',
  '• - Mitigation: Terminated unauthorized access, engaged cybersecurity specialists, implemented additional security enhancements, and offered complimentary two-year credit monitoring and identity protection services to affected customers',
  'for decision-makers to understand the current cybersecurity landscape and take appropriate actions to safeguard their organizations',
  'to restart them in a controlled manner',
  'swift response strategies to mitigate the impact of sophisticated cyberattacks',
  'This report is compiled for decision-makers to understand the current cybersecurity landscape and take appropriate actions to safeguard their organizations',
  'notification of affected customers, and advice to monitor personal information for signs of identity theft',
  'take appropriate ac

In [8]:
# End-to-end evaluation
def process_report_against_articles(
    report_text: str,
    article_paths: List[str],
    context_window: int = 1,
    top_k: int = 3,
    min_score: float = 0.0
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    report_doc = nlp(report_text)
    claims = extract_claims(report_doc)
    corpus = build_corpus(article_paths)

    match_rows = []
    overall_rows = []

    for cl in claims:
        claim_doc = nlp(cl["claim"])
        lemmas = content_lemmas(claim_doc)
        claim_lemmas = set(lemmas)
        claim_bigrams = bigrams(lemmas)

        per_article = []

        for art in corpus:
            best = best_support_span(
                claim_lemmas,
                claim_bigrams,
                art,
                context_window
            )
            match_rows.append({
                "claim": cl["claim"],
                "report_sentence_index": cl["sent_index"],
                "article_path": art["article_path"],
                "support_span": best["span_text"] if best["score"] >= min_score else None,
                "support_score": best["score"]
            })
            if best["sent_index"] is not None:
                per_article.append(best)

        per_article.sort(key=lambda x: x["score"], reverse=True)
        topk = per_article[:top_k]

        if topk:
            best1 = topk[0]
            overall_rows.append({
                "claim": cl["claim"],
                "report_sentence_index": cl["sent_index"],
                "best_article_path": best1["article_path"],
                "best_support_span": best1["span_text"],
                "best_support_score": best1["score"],
                "top_k_support": topk
            })
        else:
            overall_rows.append({
                "claim": cl["claim"],
                "report_sentence_index": cl["sent_index"],
                "best_article_path": None,
                "best_support_span": None,
                "best_support_score": 0.0,
                "top_k_support": []
            })

    return pd.DataFrame(match_rows), pd.DataFrame(overall_rows)

In [11]:
#zeroshot claims
ZSclaim_article_matches_df, ZSbest_overall_df = process_report_against_articles(
    report_text=zeroshot_report_text,
    article_paths=article_paths,
    context_window=1,   # include ±1 sentence around the best match
    top_k=3,
    min_score=0.0
)

print("🔎 claim Per-article best matches (top-1 for each claim & each article):")
display(ZSclaim_article_matches_df)

print("\n🏆 Best overall support per claim (top-1 across all articles) + top-k list:")
display(ZSbest_overall_df)

Loaded 5 articles.
🔎 claim Per-article best matches (top-1 for each claim & each article):


,claim,report_sentence_index,article_path,support_span,support_score
0,Structured Report: Automotive Industry Cyber T...,0,publication articles/Motility Software Solutio...,,0.0000
1,Structured Report: Automotive Industry Cyber T...,0,publication articles/Hyundai.pdf,,0.0000
2,Structured Report: Automotive Industry Cyber T...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1647
3,Structured Report: Automotive Industry Cyber T...,0,publication articles/Nidec.pdf,,0.0000
4,Structured Report: Automotive Industry Cyber T...,0,publication articles/Jaguar.pdf,,0.0000
...,...,...,...,...,...
865,This structured report provides a comprehensiv...,45,publication articles/Motility Software Solutio...,,0.0000
866,This structured report provides a comprehensiv...,45,publication articles/Hyundai.pdf,,0.0000
867,This structured report provides a comprehensiv...,45,publication articles/Electronics manufacturer.txt,Data I/O produces electronics used in vehicles...,0.0625
868,This structured report provides a comprehensiv...,45,publication articles/Nidec.pdf,,0.0000



🏆 Best overall support per claim (top-1 across all articles) + top-k list:


,claim,report_sentence_index,best_article_path,best_support_span,best_support_score,top_k_support
0,Structured Report: Automotive Industry Cyber T...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1647,[{'article_path': 'publication articles/Electr...
1,incidents affecting the automotive industry,0,publication articles/Electronics manufacturer.txt,Data I/O produces electronics used in vehicles...,0.0476,[{'article_path': 'publication articles/Electr...
2,"incidents affecting the automotive industry, i...",0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1947,[{'article_path': 'publication articles/Electr...
3,This report provides an overview of recent cyb...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1885,[{'article_path': 'publication articles/Electr...
4,cyberattacks,0,None,None,0.0000,[]
...,...,...,...,...,...,...
169,This structured report provides a comprehensiv...,45,publication articles/Electronics manufacturer.txt,The company admitted that the “expected costs ...,0.0714,[{'article_path': 'publication articles/Electr...
170,"incidents affecting the automotive industry, a...",45,publication articles/Electronics manufacturer.txt,Data I/O produces electronics used in vehicles...,0.0435,[{'article_path': 'publication articles/Electr...
171,affecting the automotive industry,45,publication articles/Electronics manufacturer.txt,Data I/O produces electronics used in vehicles...,0.0500,[{'article_path': 'publication articles/Electr...
172,additional context,45,None,None,0.0000,[]


In [27]:
#COT claims
COTclaim_article_matches_df, COTbest_overall_df = process_report_against_articles(
    report_text=cot_report_text,
    article_paths=article_paths,
    context_window=1,   # include ±1 sentence around the best match
    top_k=3,
    min_score=0.0
)

print("🔎 claim Per-article best matches (top-1 for each claim & each article):")
display(COTclaim_article_matches_df)

print("\n🏆 Best overall support per claim (top-1 across all articles) + top-k list:")
display(COTbest_overall_df)

Loaded 5 articles.
🔎 claim Per-article best matches (top-1 for each claim & each article):


,claim,report_sentence_index,article_path,support_span,support_score
0,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Motility Software Solutio...,,0.0000
1,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Hyundai.pdf,,0.0000
2,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.1690
3,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Nidec.pdf,,0.0000
4,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Jaguar.pdf,,0.0000
...,...,...,...,...,...
655,This report is compiled for decision-makers to...,38,publication articles/Motility Software Solutio...,,0.0000
656,This report is compiled for decision-makers to...,38,publication articles/Hyundai.pdf,,0.0000
657,This report is compiled for decision-makers to...,38,publication articles/Electronics manufacturer.txt,The company admitted that the “expected costs ...,0.0556
658,This report is compiled for decision-makers to...,38,publication articles/Nidec.pdf,,0.0000



🏆 Best overall support per claim (top-1 across all articles) + top-k list:


,claim,report_sentence_index,best_article_path,best_support_span,best_support_score,top_k_support
0,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.1690,[{'article_path': 'publication articles/Electr...
1,Jaguar Land Rover,0,None,None,0.0000,[]
2,"affecting various organizations, including Hyu...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.1147,[{'article_path': 'publication articles/Electr...
3,"Data I/O, Nidec Corporation, and Jaguar Land R...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.2121,[{'article_path': 'publication articles/Electr...
4,Cybersecurity Report Executive Summary This re...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1175,[{'article_path': 'publication articles/Electr...
...,...,...,...,...,...,...
127,take appropriate actions to safeguard their or...,38,None,None,0.0000,[]
128,to safeguard their organizations,38,None,None,0.0000,[]
129,This report is compiled for decision-makers to...,38,publication articles/Electronics manufacturer.txt,The company admitted that the “expected costs ...,0.0714,[{'article_path': 'publication articles/Electr...
130,for decision-makers to understand the current ...,38,publication articles/Electronics manufacturer.txt,Data I/O is the second company this week to re...,0.0455,[{'article_path': 'publication articles/Electr...


In [29]:
#fewshot claims
FSclaim_article_matches_df, FSbest_overall_df = process_report_against_articles(
    report_text=fewshot_report_text,
    article_paths=article_paths,
    context_window=1,   # include ±1 sentence around the best match
    top_k=3,
    min_score=0.0
)

print("🔎 claim Per-article best matches (top-1 for each claim & each article):")
display(FSclaim_article_matches_df)

print("\n🏆 Best overall support per claim (top-1 across all articles) + top-k list:")
display(FSbest_overall_df)

Loaded 5 articles.
🔎 claim Per-article best matches (top-1 for each claim & each article):


,claim,report_sentence_index,article_path,support_span,support_score
0,Executive Summary This report summarizes recen...,0,publication articles/Motility Software Solutio...,,0.0000
1,Executive Summary This report summarizes recen...,0,publication articles/Hyundai.pdf,,0.0000
2,Executive Summary This report summarizes recen...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1246
3,Executive Summary This report summarizes recen...,0,publication articles/Nidec.pdf,,0.0000
4,Executive Summary This report summarizes recen...,0,publication articles/Jaguar.pdf,,0.0000
...,...,...,...,...,...
750,to investigate breaches and implement necessar...,21,publication articles/Motility Software Solutio...,,0.0000
751,to investigate breaches and implement necessar...,21,publication articles/Hyundai.pdf,,0.0000
752,to investigate breaches and implement necessar...,21,publication articles/Electronics manufacturer.txt,Learn more.\n Recorded Future\nElectronics man...,0.0714
753,to investigate breaches and implement necessar...,21,publication articles/Nidec.pdf,,0.0000



🏆 Best overall support per claim (top-1 across all articles) + top-k list:


,claim,report_sentence_index,best_article_path,best_support_span,best_support_score,top_k_support
0,Executive Summary This report summarizes recen...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1246,[{'article_path': 'publication articles/Electr...
1,Executive Summary This report summarizes recen...,0,publication articles/Electronics manufacturer.txt,The company reported $5.9 million in sales las...,0.1151,[{'article_path': 'publication articles/Electr...
2,"Motility Software Solutions, Data I/O, Nidec C...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.1690,[{'article_path': 'publication articles/Electr...
3,Jaguar Land Rover,0,None,None,0.0000,[]
4,"Data I/O, Nidec Corporation, and Jaguar Land R...",0,publication articles/Electronics manufacturer.txt,"The Redmond, Washington-based company said the...",0.2121,[{'article_path': 'publication articles/Electr...
...,...,...,...,...,...,...
146,implement necessary security enhancements to p...,21,publication articles/Electronics manufacturer.txt,Learn more.\n Recorded Future\nElectronics man...,0.0769,[{'article_path': 'publication articles/Electr...
147,to prevent future incidents,21,publication articles/Electronics manufacturer.txt,Learn more.\n Recorded Future\nElectronics man...,0.0909,[{'article_path': 'publication articles/Electr...
148,Organizations should also consider engaging wi...,21,publication articles/Electronics manufacturer.txt,There is no timeline for a full restoration of...,0.0948,[{'article_path': 'publication articles/Electr...
149,Organizations should also consider engaging wi...,21,publication articles/Electronics manufacturer.txt,There is no timeline for a full restoration of...,0.1080,[{'article_path': 'publication articles/Electr...


In [30]:
def faithfulness_score(
    best_overall_df: pd.DataFrame,
    min_score: float = 0.2
) -> float:
    """
    Faithfulness = supported claims / total claims
    """
    if best_overall_df.empty:
        return 0.0

    total_claims = len(best_overall_df)

    supported_claims = (
        best_overall_df["best_support_score"] >= min_score
    ).sum()

    return round(supported_claims / total_claims, 4)

In [37]:
score = faithfulness_score(best_overall_df, min_score=0.2)
print("Faithfulness score:", score)

Faithfulness score: 0.106


In [38]:
def normalize_faithfulness(faithfulness_0_1: float) -> float:
    """
    Convert faithfulness score from [0,1] → [0,5]
    """
    return round(max(0.0, min(5.0, faithfulness_0_1 * 5)), 2)

In [39]:
norm = normalize_faithfulness(score)
norm

0.53

In [ ]:
# save to CSV
claim_article_matches_df.to_csv("claim_article_matches.csv", index=False, encoding="utf-8")
best_overall_df.to_csv("best_overall_support.csv", index=False, encoding="utf-8")
print("✅ Saved CSVs: claim_article_matches.csv, best_overall_support.csv")